# Critical Reproduction of a Phishing Website Detection Tutorial

**Student:** Layan Awawde  
**Course:** Data Science in Cybersecurity  
**Selected source:** Nicolas Papernot, *Detecting phishing websites using a decision tree*.

## Objective
Reproduce the tutorial claim of about 90.5% accuracy, inspect data quality, compare several models, evaluate cybersecurity-relevant metrics, and critically assess whether the author's conclusions are justified.

## 1. Data loading and inspection

In [1]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *
RANDOM_STATE=42
FEATURE_NAMES=['having_IP_Address', 'URL_Length', 'Shortining_Service', 'having_At_Symbol', 'double_slash_redirecting', 'Prefix_Suffix', 'having_Sub_Domain', 'SSLfinal_State', 'Domain_registeration_length', 'Favicon', 'port', 'HTTPS_token', 'Request_URL', 'URL_of_Anchor', 'Links_in_tags', 'SFH', 'Submitting_to_email', 'Abnormal_URL', 'Redirect', 'on_mouseover', 'RightClick', 'popUpWidnow', 'Iframe', 'age_of_domain', 'DNSRecord', 'web_traffic', 'Page_Rank', 'Google_Index', 'Links_pointing_to_page', 'Statistical_report']
df=pd.read_csv('../data/dataset.csv',header=None,names=FEATURE_NAMES+['Result'])
df.shape, df.head()

((11055, 31),
    having_IP_Address  URL_Length  ...  Statistical_report  Result
 0                 -1           1  ...                  -1      -1
 1                  1           1  ...                   1      -1
 2                  1           0  ...                  -1      -1
 3                  1           0  ...                   1      -1
 4                  1           0  ...                   1       1
 
 [5 rows x 31 columns])

In [2]:
print('Missing:',df.isna().sum().sum())
print('Exact duplicates:',df.duplicated().sum())
print('Feature-vector duplicates:',df.drop(columns='Result').duplicated().sum())
print(df['Result'].value_counts())

Missing: 0
Exact duplicates: 5206
Feature-vector duplicates: 5270
Result
 1    6157
-1    4898
Name: count, dtype: int64


**Observed:** 11055 rows and 31 columns; 5206 exact duplicate rows; no missing values. The README describes 2,456 websites, but the current CSV contains 11,055.

## 2. EDA and feature engineering analysis
All predictors are discrete indicators encoded as -1, 0, or 1. Continuous outlier methods such as z-score and IQR are therefore inappropriate. Frequency tables, class-conditional crosstabs, duplicate analysis, and Spearman correlation are more suitable.

In [3]:
display(pd.crosstab(df['SSLfinal_State'],df['Result'],normalize='index'))
display(df[FEATURE_NAMES].corr(method='spearman'))

Result,-1,1
SSLfinal_State,,
-1,0.857745,0.142255
0,0.982005,0.017995
1,0.110725,0.889275


,having_IP_Address,URL_Length,Shortining_Service,having_At_Symbol,double_slash_redirecting,Prefix_Suffix,having_Sub_Domain,SSLfinal_State,Domain_registeration_length,Favicon,port,HTTPS_token,Request_URL,URL_of_Anchor,Links_in_tags,SFH,Submitting_to_email,Abnormal_URL,Redirect,on_mouseover,RightClick,popUpWidnow,Iframe,age_of_domain,DNSRecord,web_traffic,Page_Rank,Google_Index,Links_pointing_to_page,Statistical_report
having_IP_Address,1.000000,-0.053575,0.403461,0.158699,0.397389,-0.005257,-0.079054,0.075236,-0.022739,0.087025,0.060979,0.363534,0.029773,0.101372,0.011922,-0.027871,0.077989,0.336549,-0.321181,0.084059,0.042881,0.096882,0.054694,-0.010446,-0.050733,0.004328,-0.091774,0.029153,-0.365480,-0.019103
URL_Length,-0.053575,1.000000,-0.103928,-0.079070,-0.078865,0.050731,0.002012,0.043136,-0.216549,-0.041222,0.003374,-0.097024,0.240542,-0.027557,0.052083,0.395370,-0.022868,-0.115380,0.044530,-0.049957,-0.012798,-0.050802,-0.014340,0.175272,-0.041113,0.011568,0.180999,0.002192,-0.013534,-0.071194
Shortining_Service,0.403461,-0.103928,1.000000,0.104447,0.842796,-0.080471,-0.041791,-0.060377,0.060923,0.006101,0.002201,0.757838,-0.037235,0.001468,-0.128169,-0.022016,0.049328,0.739290,-0.534530,0.062383,0.038118,0.036616,0.016581,-0.052596,0.436064,-0.051281,0.014591,0.155844,-0.222049,0.085461
having_At_Symbol,0.158699,-0.079070,0.104447,1.000000,0.086960,-0.011726,-0.058614,0.034112,0.015522,0.304899,0.364891,0.104561,0.027909,0.061087,-0.067810,-0.018989,0.370123,0.203945,-0.028160,0.279697,0.219503,0.290893,0.284410,-0.005499,-0.047872,0.030065,-0.064735,0.037061,-0.011860,-0.080357
double_slash_redirecting,0.397389,-0.078865,0.842796,0.086960,1.000000,-0.085590,-0.042953,-0.035496,0.047464,0.035100,0.025060,0.760799,-0.026368,-0.003408,-0.119682,-0.039228,0.031898,0.723724,-0.591478,0.086635,0.025863,0.054463,0.010459,-0.050107,0.431409,-0.067604,-0.003132,0.178415,-0.219146,0.070390
Prefix_Suffix,-0.005257,0.050731,-0.080471,-0.011726,-0.085590,1.000000,0.090581,0.268943,-0.096799,-0.007504,-0.022546,-0.070153,0.098675,0.345412,0.100755,-0.016348,-0.045000,-0.077620,0.016271,0.012578,-0.024868,-0.014733,-0.036904,0.074116,-0.016556,0.121126,-0.006834,0.067781,0.075783,-0.002763
having_Sub_Domain,-0.079054,0.002012,-0.041791,-0.058614,-0.042953,0.090581,1.000000,0.272061,-0.084613,-0.016558,0.005116,-0.037074,0.106590,0.238213,0.093162,0.107000,0.009091,-0.034102,0.030016,-0.017273,0.018537,-0.024970,0.010717,0.123921,0.125486,0.000758,0.119105,0.057545,-0.007179,0.082809
SSLfinal_State,0.075236,0.043136,-0.060377,0.034112,-0.035496,0.268943,0.272061,1.000000,-0.197935,-0.013160,0.030675,-0.030101,0.199586,0.564536,0.177580,0.168264,0.011180,-0.046598,-0.021206,0.025222,0.015559,-0.011215,-0.005128,0.149757,0.052558,0.278930,0.080435,0.097121,-0.006918,0.065305
Domain_registeration_length,-0.022739,-0.216549,0.060923,0.015522,0.047464,-0.096799,-0.084613,-0.197935,1.000000,0.054253,0.022478,0.059161,-0.609970,-0.161670,-0.103034,-0.131847,0.039260,0.058109,-0.016300,0.023784,0.023520,0.051410,0.004393,-0.062851,-0.010477,-0.133357,-0.059898,-0.039766,0.114382,-0.002212
Favicon,0.087025,-0.041222,0.006101,0.304899,0.035100,-0.007504,-0.016558,-0.013160,0.054253,1.000000,0.803834,0.049483,-0.004620,0.040695,-0.097984,-0.012391,0.668317,0.071848,-0.015621,0.706179,0.414382,0.939633,0.627607,-0.002628,0.088211,-0.045592,0.011699,-0.016668,-0.139265,0.300917


## 3. Original reproduction

In [4]:
X=df[FEATURE_NAMES]; y=df['Result']
model=DecisionTreeClassifier(random_state=RANDOM_STATE)
model.fit(X.iloc[:2000],y.iloc[:2000])
pred=model.predict(X.iloc[2000:])
print('Train size:',2000,'Test size:',len(pred),'Accuracy:',accuracy_score(y.iloc[2000:],pred))

Train size: 2000 Test size: 9055 Accuracy: 0.9051352843732744


Using the current file, the original-style split yields **0.9051 accuracy** on **9055** test rows, not 456.

## 4. Improved model comparison

In [5]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=RANDOM_STATE)
models={'Logistic Regression':Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))]),'Decision Tree':DecisionTreeClassifier(max_depth=8,min_samples_leaf=5,random_state=RANDOM_STATE),'Random Forest':RandomForestClassifier(n_estimators=100,min_samples_leaf=2,random_state=RANDOM_STATE)}
rows=[]
for name,m in models.items():
 m.fit(X_train,y_train); p=m.predict(X_test); s=m.predict_proba(X_test)[:,1]; tn,fp,fn,tp=confusion_matrix(y_test,p,labels=[-1,1]).ravel(); rows.append([name,accuracy_score(y_test,p),precision_score(y_test,p),recall_score(y_test,p),f1_score(y_test,p),fbeta_score(y_test,p,beta=2),matthews_corrcoef(y_test,p),roc_auc_score((y_test==1).astype(int),s),tn,fp,fn,tp])
results=pd.DataFrame(rows,columns=['Model','Accuracy','Precision','Recall','F1','F2','MCC','ROC_AUC','TN','FP','FN','TP']).set_index('Model')
display(results)

,Accuracy,Precision,Recall,F1,F2,MCC,ROC_AUC,TN,FP,FN,TP
Model,,,,,,,,,,,
Logistic Regression,0.928539,0.923441,0.950447,0.936749,0.944920,0.855137,0.980802,883,97,61,1170
Decision Tree,0.936680,0.955723,0.929326,0.942339,0.934488,0.872580,0.985616,927,53,87,1144
Random Forest,0.966079,0.964630,0.974817,0.969697,0.972763,0.931245,0.996645,936,44,31,1200


## 5. Duplicate sensitivity
Remove exact duplicates before splitting because identical rows across train and test can make performance optimistic.

In [6]:
dedup=df.drop_duplicates().reset_index(drop=True)
print('Before:',len(df),'After:',len(dedup))

Before: 11055 After: 5849


## 6. Evaluation metrics
Accuracy measures total correctness. Precision measures how many phishing alerts are correct. Recall measures how many phishing pages are caught. F1 balances precision and recall. F2 emphasizes recall. MCC summarizes all confusion-matrix cells. ROC-AUC measures ranking quality across thresholds. In phishing detection, false negatives are usually more dangerous, while false positives cause blocking and alert fatigue.

## 7. Conclusions
The tutorial claim is approximately reproducible in magnitude, but the repository is internally inconsistent and the evaluation is incomplete. A stronger study requires stratification, fixed seeds, cross-validation, duplicate-aware evaluation, multiple metrics, and temporal validation on newer phishing campaigns.